# **STEP 1 : CREATE CLASS VALUE**

In [1]:
class Value:
    def __init__(self,data,children=(),op=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(children)
        self._op = op

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    def __add__(self, other):
        other = other if isinstance(other,Value) else Value(other)
        output = Value(self.data + other.data,(self,other),'+')
        def _backward():
            self.grad += output.grad
            other.grad += output.grad
        output._backward = _backward
        return output

    def __mul__(self, other):
        other = other if isinstance(other,Value) else Value(other)
        output = Value(self.data * other.data,(self,other),'*')
        def _backward():
            self.grad += other.data * output.grad
            other.grad += self.data * output.grad
        output._backward = _backward
        return output

    def relu(self):
        output = Value(max(0,self.data),(self,),'relu')
        def _backward():
            self.grad += (1.0 if output.data > 0 else 0.0) * output.grad
        output._backward = _backward
        return output

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        self.grad = 1.0
        for v in reversed(topo):
            v._backward()

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)

    def __radd__(self, other):
        return self + other

    def __rmul__(self, other):
        return self * other

    def __rsub__(self, other):
        return other + (-self)

    def __pow__(self, n):
        out = Value(self.data ** n, (self,), f'**{n}')
        def _backward():
            self.grad += n * (self.data ** (n - 1)) * out.grad
        out._backward = _backward
        return out

    def __truediv__(self, other):
        return self * (other ** -1) if isinstance(other, Value) else self * (Value(other) ** -1)

    def exp(self):
        import math
        e = math.exp(self.data)
        out = Value(e, (self,), 'exp')
        def _backward():
            self.grad += e * out.grad
        out._backward = _backward
        return out

    def log(self):
        import math
        out = Value(math.log(self.data), (self,), 'log')
        def _backward():
            self.grad += (1.0 / self.data) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        import math
        t = math.tanh(self.data)
        out = Value(t, (self,), 'tanh')
        def _backward():
            self.grad += (1 - t ** 2) * out.grad
        out._backward = _backward
        return out
    


# **STEP 2 : MINI MLP FROM STRATCH**

In [ ]:
import random

class Neuron:
    def __init__(self, n_inputs):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(n_inputs)]
        self.b = Value(0.0)

    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.tanh()

    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self, n_inputs, n_outputs):
        self.neurons = [Neuron(n_inputs) for _ in range(n_outputs)]

    def __call__(self, x):
        return [n(x) for n in self.neurons]

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

class MLP:
    def __init__(self, sizes):
        self.layers = [Layer(sizes[i], sizes[i+1]) for i in range(len(sizes)-1)]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x[0] if len(x) == 1 else x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]


In [7]:
random.seed(42)
model = MLP([2,10,1])

xs = [[0, 0], [0, 1], [1, 0], [1, 1]]
ys = [-1, 1, 1, -1]

for step in range(100):
    preds = [model(x) for x in xs]
    loss = sum((p-y) ** 2 for p,y in zip(preds,ys))

    for p in model.parameters():
        p.grad = 0.0
    loss.backward()

    lr = 0.05
    for p in model.parameters():
        p.data -= lr * p.grad

    if step % 20 == 0:
        print(f'Step {step:3d} loss = {loss.data:4f}')

print("\nPredictions after training:")
for x, y in zip(xs, ys):
    print(f"  input={x}  target={y:2d}  pred={model(x).data:6.3f}")

Step   0 loss = 5.458855
Step  20 loss = 1.043155
Step  40 loss = 0.324136
Step  60 loss = 0.169599
Step  80 loss = 0.107859

Predictions after training:
  input=[0, 0]  target=-1  pred=-0.895
  input=[0, 1]  target= 1  pred= 0.855
  input=[1, 0]  target= 1  pred= 0.856
  input=[1, 1]  target=-1  pred=-0.844


In [9]:
def gradient_check(build_expr, x_val, h=1e-7):
    x = Value(x_val)
    y = build_expr(x)
    y.backward()
    autodiff_grad = x.grad

    y_plus = build_expr(Value(x_val + h)).data
    y_minus = build_expr(Value(x_val - h)).data
    numerical_grad = (y_plus - y_minus) / (2 * h)

    diff = abs(autodiff_grad - numerical_grad)
    return autodiff_grad, numerical_grad, diff

def expr(x):
    return (x ** 3 + x * 2 + 1).tanh()

ad, num, diff = gradient_check(expr, 0.5)
print(f"Autodiff:  {ad:.8f}")
print(f"Numerical: {num:.8f}")
print(f"Difference: {diff:.2e}")
# Difference should be < 1e-5

Autodiff:  0.15252426
Numerical: 0.15252426
Difference: 3.66e-10


# **STEP 3 : MANUAL CALCUCALTION**

In [ ]:
x1 = Value(2.0)
x2 = Value(3.0)
a  = x1 * x2
b  = a + Value(1.0)

y = b.relu()
y.backward()

print(f"y = {y.data}")        
print(f"dy/dx1 = {x1.grad}")   
print(f"dy/dx2 = {x2.grad}")

y = 7.0
dy/dx1 = 3.0
dy/dx2 = 2.0


In [12]:
import torch

x1 = torch.tensor(2.0, requires_grad=True)
x2 = torch.tensor(3.0, requires_grad=True)
a = x1 * x2
b = a + 1.0
y = torch.relu(b)
y.backward()

print(f"PyTorch dy/dx1 = {x1.grad.item()}")  # 3.0
print(f"PyTorch dy/dx2 = {x2.grad.item()}")  # 2.0

PyTorch dy/dx1 = 3.0
PyTorch dy/dx2 = 2.0


# **LATIHAN SOAL**

**1.** Tambahkan `__pow__` ke kelas `Value` agar Anda dapat menghitung `x ** n`. Verifikasi bahwa $\frac{d}{dx}(x^3)$ pada $x = 2$ sama dengan `12.0`.


In [22]:
Xa = Value(0.2)
y = Xa ** 3
y.backward()

print(f"Forward Pass : {y.data}")
print(f"Backward Pass: {Xa.grad}")

Forward Pass : 0.008000000000000002
Backward Pass: 0.12000000000000002


**2.** Tambahkan `tanh` sebagai fungsi aktivasi. Verifikasi bahwa $\tanh'(0) = 1$ dan $\tanh'(2) \approx 0.0707$.

In [25]:
X = [0, 2]

for i in range(len(X)):
    Xb = Value(X[i])
    y = Xb.tanh()
    y.backward()
    print(f"x = {X[i]}")
    print(f"Forward Pass : {y.data}")
    print(f"Backward Pass: {Xb.grad}")
    print("-" * 30)

x = 0
Forward Pass : 0.0
Backward Pass: 1.0
------------------------------
x = 2
Forward Pass : 0.9640275800758169
Backward Pass: 0.07065082485316443
------------------------------


**3.** Buat grafik komputasi untuk satu neuron: `y = relu(w1*x1 + w2*x2 + b)`. Hitung kelima gradien dan verifikasi terhadap PyTorch.

In [36]:
import torch

w1 = [Value(random.uniform(-1, 1)) for _ in range(5)]
x1 = [Value(random.uniform(-1, 1)) for _ in range(5)]
w2 = [Value(random.uniform(-1, 1)) for _ in range(5)]
x2 = [Value(random.uniform(-1, 1)) for _ in range(5)]
b = [Value(random.uniform(-1, 1)) for _ in range(5)]

wt_1 = [torch.tensor(w1[i].data, requires_grad=True) for i in range(5)]
xt_1 = [torch.tensor(x1[i].data, requires_grad=True) for i in range(5)]
wt_2 = [torch.tensor(w2[i].data, requires_grad=True) for i in range(5)]
xt_2 = [torch.tensor(x2[i].data, requires_grad=True) for i in range(5)]
bt = [b[i].data for i in range(5)]

y_value = []
y_tensor = []

for i in range(5):
    result_value = ((w1[i] * x1[i]) + (w2[i] * x2[i]) + b[i])
    result_tensor = ((wt_1[i] * xt_1[i]) + (wt_2[i] * xt_2[i]) + bt[i])

    output_value = result_value.relu()
    output_tensor = result_tensor.relu()

    output_value.backward()
    output_tensor.backward()

    y_value.append(output_value)
    y_tensor.append(output_tensor)

    print(f"\n========== NEURON {i + 1} ==========")
    print(f"Forward Value  : {output_value.data:.6f}")
    print(f"Forward PyTorch: {output_tensor.item():.6f}")
    print("\nPerbandingan Gradien:")
    print(f"{'Variabel':<10} {'Value':>12} {'PyTorch':>12} {'Status':>12}")

    gradients_value = [w1[i].grad,x1[i].grad,w2[i].grad,x2[i].grad,b[i].grad]
    gradients_tensor = [ wt_1[i].grad.item(), xt_1[i].grad.item(),wt_2[i].grad.item(), xt_2[i].grad.item(),1.0 if result_tensor.item() > 0 else 0.0]
    names = ["w1", "x1", "w2", "x2", "b"]

    for name, grad_value, grad_tensor in zip(names, gradients_value, gradients_tensor):
        status = "SAMA" if abs(grad_value - grad_tensor) < 1e-6 else "BEDA"
        print(
            f"{name:<10} {grad_value:>12.6f} "
            f"{grad_tensor:>12.6f} {status:>12}"
        )


========== NEURON 1 ==========
Forward Value  : 1.838470
Forward PyTorch: 1.838470

Perbandingan Gradien:
Variabel          Value      PyTorch       Status
w1             0.717187     0.717187         SAMA
x1             0.621499     0.621499         SAMA
w2             0.728706     0.728706         SAMA
x2             0.590691     0.590691         SAMA
b              1.000000     1.000000         SAMA

========== NEURON 2 ==========
Forward Value  : 0.000000
Forward PyTorch: 0.000000

Perbandingan Gradien:
Variabel          Value      PyTorch       Status
w1             0.000000    -0.000000         SAMA
x1             0.000000    -0.000000         SAMA
w2             0.000000     0.000000         SAMA
x2             0.000000    -0.000000         SAMA
b              0.000000     0.000000         SAMA

========== NEURON 3 ==========
Forward Value  : 1.663232
Forward PyTorch: 1.663232

Perbandingan Gradien:
Variabel          Value      PyTorch       Status
w1             0.633173     0

**4.** Implementasikan *autodiff* mode maju (*forward-mode*) menggunakan bilangan ganda (*dual numbers*). Buat sebuah kelas `Dual` dan verifikasi bahwa kelas tersebut memberikan turunan yang sama dengan mesin mode terbalik (*reverse-mode*) Anda.